Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import random


System Data – Define Buses,Define Generators

In [ ]:
# Generator data (for 2 generators, e.g. buses 1 and 2)
generators = pd.DataFrame({
    'gen_id': [1, 2],
    'bus_id': [1, 2],
    'P_min': [0, 0],
    'P_max': [232.4, 40],
    'cost_c2': [0.0430293, 0.25],
    'cost_c1': [20, 20],
    'cost_c0': [0, 0]
})

# Bus active power demand (Pd): IEEE 14-bus test case
buses = pd.DataFrame({
    'bus_id': range(1, 15),
    'Pd': [0, 21.7, 94.2, 47.8, 7.6, 11.2, 0, 0, 29.5, 9.0, 3.5, 6.1, 13.5, 14.9]
})

# Compute total system load
Pd_total = buses['Pd'].sum()
print(f"Total load on system: {Pd_total:.2f} MW")


Total load on system: 259.00 MW


In [ ]:
# Generator dispatch cost (quadratic)
def total_generation_cost(gen_P):
    total = 0
    for i, P in enumerate(gen_P):
        coef = generators.iloc[i]
        total += coef['cost_c2'] * P**2 + coef['cost_c1'] * P + coef['cost_c0']
    return total

# Is solution feasible? (respect gen min/max, meet total load)
def is_feasible(gen_P):
    # Check gen limits
    for i, P in enumerate(gen_P):
        if P < generators.iloc[i]['P_min'] or P > generators.iloc[i]['P_max']:
            return False
    # Check total generation >= demand
    if sum(gen_P) < Pd_total:
        return False
    return True

# Add heavy penalty for infeasible
def fitness(gen_P):
    penalty = 1e6 if not is_feasible(gen_P) else 0
    return total_generation_cost(gen_P) + penalty

# Fix individuals: Proportionally scale up to meet load (and clamp to max)
def repair_individual(ind):
    gen_sum = sum(ind)
    if gen_sum <= 1e-6:  # avoids div0
        ind = [generators.iloc[i]['P_max'] for i in range(len(ind))]
        return ind
    if gen_sum < Pd_total:
        scale = Pd_total / gen_sum
        ind = [min(max(generators.iloc[i]['P_min'], ind[i] * scale), generators.iloc[i]['P_max']) for i in range(len(ind))]
    return ind


 Objective and Constraint Functions

In [ ]:
def total_generation_cost(gen_P):
    total = 0
    for i, P in enumerate(gen_P):
        coef = generators.iloc[i]
        total += coef['cost_c2'] * P * P + coef['cost_c1'] * P + coef['cost_c0']
    return total

def is_feasible(gen_P):
    # Generator limits
    for i, P in enumerate(gen_P):
        if P < generators.iloc[i]['P_min'] or P > generators.iloc[i]['P_max']:
            return False
    # Power balance
    Pd_total = buses['Pd'].sum()
    if np.sum(gen_P) < Pd_total:
        return False
    return True

def fitness(gen_P):
    penalty = 1e6 if not is_feasible(gen_P) else 0
    return total_generation_cost(gen_P) + penalty


In [ ]:
# Cell: Define branch data
branches = pd.DataFrame({
    'from_bus': [1, 2, 2, 1, 5, 2, 6, 6, 6, 9, 9, 4, 4, 5, 13, 13, 1, 1, 3, 11],
    'to_bus': [2, 3, 4, 5, 6, 7, 11, 12, 13, 10, 14, 12, 11, 10, 14, 9, 1, 2, 5, 13],
    'r': np.random.uniform(0.01, 0.1, 20),    # Placeholder random resistance
    'x': np.random.uniform(0.01, 0.2, 20),    # Placeholder random reactance
    'rateA': [130]*20
})
branches.head()

,from_bus,to_bus,r,x,rateA
0,1,2,0.070262,0.136949,130
1,2,3,0.012626,0.184414,130
2,2,4,0.047157,0.152837,130
3,1,5,0.075117,0.040389,130
4,5,6,0.038573,0.141696,130


In [ ]:
print("Number of buses:", len(buses))
print("Number of generators:", len(generators))
print("Total load (MW):", buses['Pd'].sum())

Number of buses: 14
Number of generators: 2
Total load (MW): 259.0


Genetic Algorithem

In [ ]:
import random
import pandas as pd

# GA parameters
POP_SIZE = 50
NUM_GENS = 100
MUTATION_RATE = 0.1

def random_individual():
    ind = [random.uniform(generators.iloc[i]['P_min'], generators.iloc[i]['P_max']) for i in range(len(generators))]
    return repair_individual(ind)

def create_population():
    return [random_individual() for _ in range(POP_SIZE)]

def select_parents(population):
    sorted_pop = sorted(population, key=fitness)
    return random.choices(sorted_pop[:POP_SIZE // 2], k=2)

def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    return parent1[:point] + parent2[point:]

def mutate(ind):
    child = ind[:]
    if random.random() < MUTATION_RATE:
        i = random.randint(0, len(child) - 1)
        child[i] = random.uniform(generators.iloc[i]['P_min'], generators.iloc[i]['P_max'])
    return repair_individual(child)
def run_ga():
    population = create_population()
    best_costs = []
    for generation in range(NUM_GENS):
        new_population = []
        for _ in range(POP_SIZE // 2):
            p1, p2 = select_parents(population)
            child1 = mutate(crossover(p1, p2))
            child2 = mutate(crossover(p2, p1))
            new_population += [child1, child2]
        population = new_population
        best = min(population, key=fitness)
        best_costs.append(fitness(best))
        print(f"Generation {generation+1}: Best Cost = {fitness(best):.4f}")
        if is_feasible(best) and fitness(best) < 1e5:
            break
    return best, best_costs

best_solution, cost_history = run_ga()
print("\nBest Generator Outputs (MW):", best_solution)
print("Total Generation Cost: ", total_generation_cost(best_solution))
print("Feasible Solution:", is_feasible(best_solution))


Generation 1: Best Cost = 7642.5937

Best Generator Outputs (MW): [np.float64(220.9685796932888), np.float64(38.03142030671121)]
Total Generation Cost:  7642.593735155314
Feasible Solution: True


Tabu Search

In [ ]:
import pandas as pd
import numpy as np
import random
import copy

def cost_with_penalty(gen_P):
    total = 0
    for i, P in enumerate(gen_P):
        coef = generators.iloc[i]
        total += coef['cost_c2'] * P * P + coef['cost_c1'] * P + coef['cost_c0']
    # Hard constraint: generator output bounds & total generation meets load
    feasible = True
    for i, P in enumerate(gen_P):
        if P < generators.iloc[i]['P_min'] or P > generators.iloc[i]['P_max']:
            feasible = False
    Pd_total = buses['Pd'].sum()
    if np.sum(gen_P) < Pd_total:
        feasible = False
    penalty = 1e6 if not feasible else 0
    return total + penalty


TABU_TENURE = 10
MAX_ITER = 100
NEIGHBORS = 20

# Random individual
def random_solution():
    return [random.uniform(generators.iloc[i]['P_min'], generators.iloc[i]['P_max']) for i in range(len(generators))]

# Generate a neighbor by perturbing one generator
def generate_neighbor(sol):
    neighbor = sol[:]
    i = random.randint(0, len(sol)-1)
    # Small perturbation
    step = random.uniform(-2,2)
    neighbor[i] = min(max(generators.iloc[i]['P_min'], neighbor[i]+step), generators.iloc[i]['P_max'])
    return neighbor

# Tabu structure as a list of previous moves (solutions or indices changed)

def tabu_search():
    # Start with a feasible/random solution
    current = random_solution()
    while not is_feasible(current):
        current = random_solution()
    best = current[:]
    best_cost = cost_with_penalty(best)
    tabu_list = []

    for iteration in range(MAX_ITER):
        # Generate multiple neighbors
        neighborhood = [generate_neighbor(current) for _ in range(NEIGHBORS)]
        # Filter out tabu
        filtered = [n for n in neighborhood if n not in tabu_list]
        # If all neighbors are tabu, allow them
        if not filtered:
            filtered = neighborhood
        # Choose best neighbor
        filtered.sort(key=cost_with_penalty)
        best_nb = filtered[0]
        best_nb_cost = cost_with_penalty(best_nb)
        # Aspiration: override tabu if best_nb improves global best
        if best_nb_cost < best_cost:
            best = best_nb[:]
            best_cost = best_nb_cost
        # Update Tabu List
        tabu_list.append(current)
        if len(tabu_list) > TABU_TENURE:
            tabu_list.pop(0)
        current = best_nb # Move to best neighbor
        print(f"Iteration {iteration+1}: Best Cost = {best_cost}")
        if best_cost < 1e5:
            break
    return best, best_cost

# Run Tabu Search
best_sol, best_cost = tabu_search()
print("\nBest Generator Outputs (MW):", best_sol)
print("Total Generation Cost:", total_generation_cost(best_sol))
print("Feasible Solution:", is_feasible(best_sol))



Iteration 1: Best Cost = 8019.821985435959

Best Generator Outputs (MW): [np.float64(230.58822373760492), np.float64(37.97819332681091)]
Total Generation Cost: 8019.821985435959
Feasible Solution: True


Simulated Annealing

In [ ]:
import math

# Generate a random feasible solution
def random_solution():
    sol = [random.uniform(generators.iloc[i]['P_min'], generators.iloc[i]['P_max']) for i in range(len(generators))]
    while not is_feasible(sol):
        sol = [random.uniform(generators.iloc[i]['P_min'], generators.iloc[i]['P_max']) for i in range(len(generators))]
    return sol

# Small random change for neighbor
def generate_neighbor(sol):
    neighbor = sol[:]
    i = random.randint(0, len(sol)-1)
    step = random.uniform(-2, 2)
    neighbor[i] = min(max(generators.iloc[i]['P_min'], neighbor[i]+step), generators.iloc[i]['P_max'])
    return neighbor

# Simulated Annealing main loop

def simulated_annealing(max_iter=100, init_temp=1000, cooling_rate=0.95):
    current = random_solution()
    best = current[:]
    best_cost = total_generation_cost(best) if is_feasible(best) else float('inf')
    temp = init_temp

    for iteration in range(max_iter):
        neighbor = generate_neighbor(current)
        neighbor_cost = total_generation_cost(neighbor) if is_feasible(neighbor) else best_cost + 1e6
        delta = neighbor_cost - total_generation_cost(current)
        # Accept new solution based on cost difference and temperature
        if delta < 0 or random.random() < math.exp(-delta / temp):
            current = neighbor[:]
            if is_feasible(current) and neighbor_cost < best_cost:
                best = neighbor[:]
                best_cost = neighbor_cost
        temp *= cooling_rate
        print(f"Iteration {iteration+1}: Best Cost = {best_cost} | Temperature = {temp:.2f}")
        if best_cost < 1e5:  # Good enough cutoff
            break
    return best, best_cost

# Run Simulated Annealing
sa_best_sol, sa_best_cost = simulated_annealing()
print("\nBest Generator Outputs (MW):", sa_best_sol)
print("Total Generation Cost:", total_generation_cost(sa_best_sol))
print("Feasible Solution:", is_feasible(sa_best_sol))


Iteration 1: Best Cost = 8002.118559697576 | Temperature = 950.00

Best Generator Outputs (MW): [np.float64(230.33741333921557), np.float64(37.78012189800453)]
Total Generation Cost: 8002.118559697576
Feasible Solution: True


In [ ]:
import pandas as pd

# Take your actual outputs below; replace these assignments if needed:
best_solution_GA = [220.2591302620332, 38.74086973796681]
best_solution_Tabu = [225.33267244485262, 37.8043037290284]
best_solution_SA = [230.3605890119338, 31.502300855560197]

def total_generation_cost(gen_P):
    total = 0
    for i, P in enumerate(gen_P):
        coef = generators.iloc[i]
        total += coef['cost_c2'] * P * P + coef['cost_c1'] * P + coef['cost_c0']
    return total

def is_feasible(gen_P):
    for i, P in enumerate(gen_P):
        if P < generators.iloc[i]['P_min'] or P > generators.iloc[i]['P_max']:
            return False
    Pd_total = buses['Pd'].sum()
    if sum(gen_P) < Pd_total:
        return False
    return True

def summarise_results():
    results = {
        "Genetic Algorithm": [best_solution_GA, total_generation_cost(best_solution_GA), is_feasible(best_solution_GA)],
        "Tabu Search": [best_solution_Tabu, total_generation_cost(best_solution_Tabu), is_feasible(best_solution_Tabu)],
        "Simulated Annealing": [best_solution_SA, total_generation_cost(best_solution_SA), is_feasible(best_solution_SA)]
    }
    summary_df = pd.DataFrame(results, index=["Best Outputs (MW)", "Total Cost", "Feasible?"]).T
    print(summary_df)

# Run summary table display
summarise_results()


                                           Best Outputs (MW)   Total Cost  \
Genetic Algorithm     [220.2591302620332, 38.74086973796681]  7642.740842   
Tabu Search           [225.33267244485262, 37.8043037290284]  7804.835541   
Simulated Annealing  [230.3605890119338, 31.502300855560197]  7768.749413   

                    Feasible?  
Genetic Algorithm        True  
Tabu Search              True  
Simulated Annealing      True  
